<a href="https://colab.research.google.com/github/kingpentes/Kelompok6-deepLearning-tubes/blob/main/Kelompok6_DeepLearning_Paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install library utama untuk CDAD
# pytorch-lightning: training framework
# omegaconf: baca file yaml config
# einops, timm: utility arsitektur
# taming-transformers: dibutuhkan ldm/cdad
# open_clip_torch: CLIP encoder untuk text conditioning

!pip install pytorch-lightning==1.9.5 \
             omegaconf \
             einops \
             timm \
             scipy \
             scikit-learn \
             taming-transformers-rom1504 \
             open_clip_torch \
             transformers==4.44.0 \
             diffusers==0.30.0 \
             accelerate==0.34.0 \
             --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00


In [ ]:
# #konfigurasi
# import torch
# import os

# # Semua hyperparameter di satu tempat
# # Sesuai paper Section 4 (Implementation Details)

# DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# BASE_PATH  = '/content/drive/MyDrive/deep_learning/tubes_DL'
# REF_DIR    = os.path.join(BASE_PATH, 'referensi')  # folder repo CDAD

# IMG_SIZE   = 256    # semua gambar diresize ke 256x256
# BATCH_SIZE = 2
# VAL_SPLIT  = 0.2
# SEED       = 42

# # Epoch
# # Paper asli: base=500, incremental=100
# # Versi Colab (hemat waktu): base=100, incremental=30
# EPOCH_BASE        = 100
# EPOCH_INCREMENTAL = 30

# LR_BASE           = 1e-5
# LR_INCREMENTAL    = 5e-6   # lebih kecil untuk incremental

# DDIM_STEPS  = 10     #  DDIM 10 steps saat inference
# THRESHOLD   = 0.999  # gamma_th untuk iSVD
# MAX_BATCHES_SVD = 20 # jumlah batch untuk hitung significant representation

# print(f'Device    : {DEVICE}')
# if torch.cuda.is_available():
#     print(f'GPU       : {torch.cuda.get_device_name(0)}')
#     print(f'VRAM      : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
# print(f'BASE_PATH : {BASE_PATH}')
# print(f'REF_DIR   : {REF_DIR}')

In [ ]:
# import sys

# # Repo CDAD (One-for-More) harus sudah ada di Google Drive
# # Clone dari: https://github.com/FuNz-0/One-for-More

# if os.path.exists(REF_DIR):
#     sys.path.insert(0, REF_DIR)
#     os.chdir(REF_DIR)
#     print(f'REF_DIR ditemukan: {REF_DIR}')
#     print(f'Working directory: {os.getcwd()}')
#     print('Isi folder:')
#     for item in sorted(os.listdir(REF_DIR)):
#         print(f'  {item}')
# else:
#     print(f'FOLDER TIDAK DITEMUKAN: {REF_DIR}')
#     print('Pastikan sudah clone repo CDAD ke folder referensi di Drive.')
#     print('Jalankan cell berikut untuk clone otomatis.')

In [ ]:
# # base.ckpt = pre-trained Stable Diffusion v1.5 Dibutuhkan sebagai inisialisasi model CDAD
# # CDAD is based on pre-trained Stable Diffusion with a pre-trained VAE

# os.makedirs(os.path.join(REF_DIR, 'models'), exist_ok=True)
# base_ckpt_path = os.path.join(REF_DIR, 'models', 'base.ckpt')

# if not os.path.exists(base_ckpt_path):
#     print('Mendownload base.ckpt (~4GB)...')
#     print('Ini akan memakan waktu beberapa menit.')
#     os.system(
#         f'wget -q --show-progress -O "{base_ckpt_path}" '
#         f'https://huggingface.co/runwayml/stable-diffusion-v1-5/'
#         f'resolve/main/v1-5-pruned-emaonly.ckpt'
#     )
#     print('Download selesai.')
# else:
#     print(f'base.ckpt sudah ada di: {base_ckpt_path}')
#     size_gb = os.path.getsize(base_ckpt_path) / 1e9
#     print(f'Ukuran file: {size_gb:.2f} GB')

In [ ]:
# import os
# import shutil

# BASE_PATH = '/content/drive/MyDrive/deep_learning/tubes_DL'

# otak_path  = os.path.join(BASE_PATH, 'otak', 'brain_tumor_dataset')
# rahim_path = os.path.join(BASE_PATH, 'rahim')

# train_tumor_path   = os.path.join(BASE_PATH, 'train_tumor')
# testing_otak_path  = os.path.join(BASE_PATH, 'testing_otak')
# train_rahim_path   = os.path.join(BASE_PATH, 'train_rahim')
# testing_rahim_path = os.path.join(BASE_PATH, 'testing_rahim')

# IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}

# # Buat folder kalau belum ada
# for path in [train_tumor_path, testing_otak_path,
#              train_rahim_path, testing_rahim_path]:
#     os.makedirs(path, exist_ok=True)

# # HELPER
# def copy_images_to_class(src, dst_root, class_name, label=''):
#     if not os.path.exists(src):
#         print(f'  TIDAK DITEMUKAN: {src}')
#         return 0
#     dst = os.path.join(dst_root, class_name)
#     os.makedirs(dst, exist_ok=True)
#     count = 0
#     for root, dirs, files in os.walk(src):
#         dirs[:] = [d for d in dirs if not d.startswith('.')]
#         for f in files:
#             ext = os.path.splitext(f)[1].lower()
#             if ext not in IMAGE_EXT:
#                 continue
#             src_f = os.path.join(root, f)
#             dst_f = os.path.join(dst, f)
#             if os.path.exists(dst_f):
#                 name, ext2 = os.path.splitext(f)
#                 dst_f = os.path.join(dst, f'{name}_{label}{ext2}')
#             shutil.copy2(src_f, dst_f)
#             count += 1
#     print(f'  [{label}] {count} gambar -> {dst}')
#     return count

# # PROSES OTAK
# print('=== OTAK ===')
# copy_images_to_class(os.path.join(otak_path, 'healthy'),
#                      train_tumor_path,  class_name='normal',  label='healthy_train')
# copy_images_to_class(os.path.join(otak_path, 'healthy'),
#                      testing_otak_path, class_name='normal',  label='healthy_test')
# for cls in ['glioma', 'meningioma', 'pituitary']:
#     copy_images_to_class(os.path.join(otak_path, cls),
#                          testing_otak_path, class_name='anomaly', label=cls)

# # PROSES RAHIM
# print('\n=== RAHIM ===')
# for cls in ['im_Parabasal', 'im_Superficial-Intermediate']:
#     copy_images_to_class(os.path.join(rahim_path, cls),
#                          train_rahim_path,   class_name='normal', label=f'{cls}_train')
#     copy_images_to_class(os.path.join(rahim_path, cls),
#                          testing_rahim_path, class_name='normal', label=f'{cls}_test')
# for cls in ['im_Dyskeratotic', 'im_Koilocytotic', 'im_Metaplastic']:
#     copy_images_to_class(os.path.join(rahim_path, cls),
#                          testing_rahim_path, class_name='anomaly', label=cls)

# # RINGKASAN
# print('\n=== RINGKASAN ===')
# for folder_label, path in [('train_tumor',   train_tumor_path),
#                             ('testing_otak',  testing_otak_path),
#                             ('train_rahim',   train_rahim_path),
#                             ('testing_rahim', testing_rahim_path)]:
#     subfolders = sorted([d for d in os.listdir(path)
#                          if os.path.isdir(os.path.join(path, d))])
#     print(f'  {folder_label}/ -> subfolder: {subfolders}')
#     for sf in subfolders:
#         n = sum(1 for r, _, fs in os.walk(os.path.join(path, sf))
#                 for f in fs if os.path.splitext(f)[1].lower() in IMAGE_EXT)
#         print(f'    {sf:10s}: {n} gambar')
# print('\nSelesai!')

In [ ]:
import torch
import os

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_PATH = '/content/drive/MyDrive/deep_learning/tubes_DL'
REF_DIR   = os.path.join(BASE_PATH, 'referensi')

IMG_SIZE          = 256
BATCH_SIZE        = 1      # dikurangi untuk hemat VRAM
VAL_SPLIT         = 0.2
SEED              = 42
MAX_TRAIN_SAMPLES = 150    # pangkas data training

EPOCH_BASE        = 100    # paper: 500, Colab: 100
EPOCH_INCREMENTAL = 30     # paper: 100, Colab: 30

LR_BASE           = 1e-5
LR_INCREMENTAL    = 5e-6

DDIM_STEPS        = 10     # paper Section 4: DDIM 10 steps
THRESHOLD         = 0.999  # gamma_th untuk iSVD (paper Section 4.6)
MAX_BATCHES_SVD   = 15

# Path folder yang sudah ada (gambar sudah dicopy sebelumnya)
train_tumor_path   = os.path.join(BASE_PATH, 'train_tumor')
testing_otak_path  = os.path.join(BASE_PATH, 'testing_otak')
train_rahim_path   = os.path.join(BASE_PATH, 'train_rahim')
testing_rahim_path = os.path.join(BASE_PATH, 'testing_rahim')

print(f'Device    : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU       : {torch.cuda.get_device_name(0)}')
    print(f'VRAM      : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'BASE_PATH : {BASE_PATH}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Max train : {MAX_TRAIN_SAMPLES} gambar per task')
print()

# Cek folder sudah ada dan berisi gambar
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}
print('=== Cek folder dataset ===')
for label, path in [('train_tumor',   train_tumor_path),
                    ('testing_otak',  testing_otak_path),
                    ('train_rahim',   train_rahim_path),
                    ('testing_rahim', testing_rahim_path)]:
    if not os.path.exists(path):
        print(f'  WARNING: {label} tidak ditemukan! -> {path}')
        continue
    n = sum(1 for r, _, fs in os.walk(path)
            for f in fs if os.path.splitext(f)[1].lower() in IMAGE_EXT)
    subfolders = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
    print(f'  {label:20s}: {n} gambar, subfolder: {subfolders}')


Device    : cuda
GPU       : Tesla T4
VRAM      : 15.6 GB
BASE_PATH : /content/drive/MyDrive/deep_learning/tubes_DL
Batch size: 1
Max train : 150 gambar per task

=== Cek folder dataset ===
  train_tumor         : 2000 gambar, subfolder: ['normal']
  testing_otak        : 7023 gambar, subfolder: ['normal', 'anomaly']
  train_rahim         : 1723 gambar, subfolder: ['normal']
  testing_rahim       : 4886 gambar, subfolder: ['normal', 'anomaly']


In [ ]:
import sys
import os

BASE_PATH = '/content/drive/MyDrive/deep_learning/tubes_DL'
REF_DIR   = os.path.join(BASE_PATH, 'referensi')

if os.path.exists(REF_DIR):
    sys.path.insert(0, REF_DIR)
    os.chdir(REF_DIR)
    print(f'REF_DIR ditemukan: {REF_DIR}')
    print(f'Working directory : {os.getcwd()}')
else:
    print(f'FOLDER TIDAK DITEMUKAN: {REF_DIR}')
    print('Clone dulu: git clone https://github.com/FuNz-0/One-for-More')

# Patch amn.py: ganti hardcode .cuda() supaya bisa jalan di CPU juga
# Error asli: AssertionError: Torch not compiled with CUDA enabled
amn_path = os.path.join(REF_DIR, 'cdm', 'amn.py')
if os.path.exists(amn_path):
    with open(amn_path, 'r') as f:
        amn_code = f.read()
    if '.cuda()' in amn_code:
        amn_code = amn_code.replace(
            '.cuda()',
            '.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))'
        )
        with open(amn_path, 'w') as f:
            f.write(amn_code)
        print('amn.py di-patch: .cuda() -> .to(DEVICE)')
    else:
        print('amn.py sudah bersih, tidak perlu patch.')
else:
    print(f'amn.py tidak ditemukan di: {amn_path}')


REF_DIR ditemukan: /content/drive/MyDrive/deep_learning/tubes_DL/referensi
Working directory : /content/drive/.shortcut-targets-by-id/1o9G1PTaZm4XCeFIqq2xtVJi2X1HBs-yp/deep_learning/tubes_DL/referensi
amn.py sudah bersih, tidak perlu patch.


In [ ]:
#Download base checkpoint Stable Diffusion v1.5
import os

BASE_PATH = '/content/drive/MyDrive/deep_learning/tubes_DL'
REF_DIR   = os.path.join(BASE_PATH, 'referensi')

os.makedirs(os.path.join(REF_DIR, 'models'), exist_ok=True)
base_ckpt_path = os.path.join(REF_DIR, 'models', 'base.ckpt')

if not os.path.exists(base_ckpt_path):
    print('Mendownload base.ckpt (~4GB)...')
    os.system(
        f'wget -q --show-progress -O "{base_ckpt_path}" '
        f'https://huggingface.co/runwayml/stable-diffusion-v1-5/'
        f'resolve/main/v1-5-pruned-emaonly.ckpt'
    )
    print('Download selesai.')
else:
    size_gb = os.path.getsize(base_ckpt_path) / 1e9
    print(f'base.ckpt sudah ada. Ukuran: {size_gb:.2f} GB')


base.ckpt sudah ada. Ukuran: 4.27 GB


In [ ]:
#DataLoader (langsung dari folder, batch=2, 150 data)
import torch
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import datasets, transforms
import os

BASE_PATH         = '/content/drive/MyDrive/deep_learning/tubes_DL'
DEVICE            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE          = 256
BATCH_SIZE        = 2
VAL_SPLIT         = 0.2
SEED              = 42
MAX_TRAIN_SAMPLES = 150

train_tumor_path   = os.path.join(BASE_PATH, 'train_tumor')
testing_otak_path  = os.path.join(BASE_PATH, 'testing_otak')
train_rahim_path   = os.path.join(BASE_PATH, 'train_rahim')
testing_rahim_path = os.path.join(BASE_PATH, 'testing_rahim')

class PatchPerturbation:
    """Patch Perturbation: bagian AMN (Section 3.4 paper).
    Noise ditempel di patch acak agar model fokus
    rekonstruksi area normal, bukan area anomali."""
    def __init__(self, patch_size=32, num_patches=3):
        self.patch_size  = patch_size
        self.num_patches = num_patches

    def __call__(self, img_tensor):
        perturbed = img_tensor.clone()
        _, H, W   = img_tensor.shape
        for _ in range(self.num_patches):
            x = torch.randint(0, max(1, W - self.patch_size), (1,)).item()
            y = torch.randint(0, max(1, H - self.patch_size), (1,)).item()
            noise = torch.randn(3, self.patch_size, self.patch_size) * 0.5
            perturbed[:, y:y+self.patch_size, x:x+self.patch_size] = noise
        return perturbed

patch_perturb = PatchPerturbation(patch_size=32, num_patches=3)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

class CDADDatasetWrapper(Dataset):
    """Bungkus dataset ke format dict CDAD:
    dict(jpg=normal_image, txt=' ', hint=perturbed_image)
    txt pakai spasi ' ' sesuai paper Section 4."""
    def __init__(self, base_dataset, perturb_fn=None):
        self.base    = base_dataset
        self.perturb = perturb_fn

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        perturbed  = self.perturb(img) if self.perturb else img.clone()
        return {'jpg': img, 'txt': ' ', 'hint': perturbed, 'label': label}

def buat_dataloader(train_path, test_path, nama_task):
    print(f'\n{"="*50}\nTASK: {nama_task}\n{"="*50}')

    full_train      = datasets.ImageFolder(train_path, transform=train_transform)
    total_available = len(full_train)
    n_pakai         = min(MAX_TRAIN_SAMPLES, total_available)

    generator = torch.Generator().manual_seed(SEED)
    indices   = torch.randperm(total_available, generator=generator)[:n_pakai].tolist()
    subset    = Subset(full_train, indices)

    print(f'  Total tersedia : {total_available} gambar')
    print(f'  Yang dipakai   : {n_pakai} gambar')

    n_val   = int(n_pakai * VAL_SPLIT)
    n_train = n_pakai - n_val
    train_set, val_set = random_split(
        subset, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_cdad = CDADDatasetWrapper(train_set, patch_perturb)
    val_cdad   = CDADDatasetWrapper(val_set,   patch_perturb)
    test_set   = datasets.ImageFolder(test_path, transform=test_transform)

    train_loader = DataLoader(train_cdad, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_cdad,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_set,   batch_size=1,
                              shuffle=False, num_workers=2, pin_memory=True)

    print(f'  Kelas train  : {full_train.classes}')
    print(f'  Kelas test   : {test_set.classes}')
    print(f'  Train        : {n_train} | Val: {n_val} | Test: {len(test_set)}')
    print(f'  Batch size   : {BATCH_SIZE}')
    return train_loader, val_loader, test_loader

# Buat DataLoader kedua task
# Training tetap satu-satu: otak dulu baru rahim
train_otak,  val_otak,  test_otak  = buat_dataloader(
    train_tumor_path, testing_otak_path, 'OTAK')
train_rahim, val_rahim, test_rahim = buat_dataloader(
    train_rahim_path, testing_rahim_path, 'RAHIM')



TASK: OTAK
  Total tersedia : 2000 gambar
  Yang dipakai   : 150 gambar
  Kelas train  : ['normal']
  Kelas test   : ['anomaly', 'normal']
  Train        : 120 | Val: 30 | Test: 7023
  Batch size   : 2

TASK: RAHIM
  Total tersedia : 1723 gambar
  Yang dipakai   : 150 gambar
  Kelas train  : ['normal']
  Kelas test   : ['anomaly', 'normal']
  Train        : 120 | Val: 30 | Test: 4886
  Batch size   : 2


In [ ]:
import os

REF_DIR     = '/content/drive/MyDrive/deep_learning/tubes_DL/referensi'
models_path = os.path.join(REF_DIR, 'models')

# Cari yaml yang tersedia — cdad_mvtec.yaml atau cdad_visa.yaml
yaml_path = None
for candidate in ['cdad_mvtec.yaml', 'cdad_visa.yaml']:
    p = os.path.join(models_path, candidate)
    if os.path.exists(p):
        yaml_path = p
        break

if yaml_path is None:
    print('yaml belum ada — akan di-copy otomatis saat Cell 14 dijalankan.')
else:
    print(f'Menggunakan: {os.path.basename(yaml_path)}')
    with open(yaml_path, 'r') as f:
        print(f.read())


Menggunakan: cdad_mvtec.yaml
model:
  params:
    channels: 4
    cond_stage_config:
      target: ldm.modules.encoders.modules.FrozenCLIPEmbedder
    cond_stage_key: txt
    cond_stage_trainable: false
    conditioning_key: crossattn
    control_key: hint
    distance: 'eucl'
    layers:
      - 1
      - 2
      - 3
    control_stage_config:
      params:
        hint_channels: 3
        image_size: 32
        model_channels: 320
        neighbor_size: [7, 7]
      target: cdm.amn.AMN
    first_stage_config:
      params:
        ddconfig:
          attn_resolutions: []
          ch: 128
          ch_mult:
          - 1
          - 2
          - 4
          - 4
          double_z: true
          dropout: 0.0
          in_channels: 3
          num_res_blocks: 2
          out_ch: 3
          resolution: 256
          z_channels: 4
        embed_dim: 4
        lossconfig:
          target: torch.nn.Identity
        monitor: val/rec_loss
      target: ldm.models.autoencoder.AutoencoderKL

In [ ]:
import os

REF_DIR     = '/content/drive/MyDrive/deep_learning/tubes_DL/referensi'
models_path = os.path.join(REF_DIR, 'models')

# Cari yaml yang tersedia — cdad_mvtec.yaml atau cdad_visa.yaml
yaml_path = None
for candidate in ['cdad_mvtec.yaml', 'cdad_visa.yaml']:
    p = os.path.join(models_path, candidate)
    if os.path.exists(p):
        yaml_path = p
        break

if yaml_path is None:
    print('yaml belum ada — akan di-copy otomatis saat Cell 14 dijalankan.')
else:
    print(f'Menggunakan: {os.path.basename(yaml_path)}')
    with open(yaml_path, 'r') as f:
        print(f.read())


Menggunakan: cdad_mvtec.yaml
model:
  params:
    channels: 4
    cond_stage_config:
      target: ldm.modules.encoders.modules.FrozenCLIPEmbedder
    cond_stage_key: txt
    cond_stage_trainable: false
    conditioning_key: crossattn
    control_key: hint
    distance: 'eucl'
    layers:
      - 1
      - 2
      - 3
    control_stage_config:
      params:
        hint_channels: 3
        image_size: 32
        model_channels: 320
        neighbor_size: [7, 7]
      target: cdm.amn.AMN
    first_stage_config:
      params:
        ddconfig:
          attn_resolutions: []
          ch: 128
          ch_mult:
          - 1
          - 2
          - 4
          - 4
          double_z: true
          dropout: 0.0
          in_channels: 3
          num_res_blocks: 2
          out_ch: 3
          resolution: 256
          z_channels: 4
        embed_dim: 4
        lossconfig:
          target: torch.nn.Identity
        monitor: val/rec_loss
      target: ldm.models.autoencoder.AutoencoderKL

In [ ]:
#clone ulang repo
import os
import sys

BASE_PATH = '/content/drive/MyDrive/deep_learning/tubes_DL'
REF_DIR   = os.path.join(BASE_PATH, 'referensi')

# Clone ulang repo ke folder sementara
TEMP_DIR = '/content/cdad_temp'
if os.path.exists(TEMP_DIR):
    os.system(f'rm -rf {TEMP_DIR}')

print('Cloning repo CDAD...')
ret = os.system(f'git clone https://github.com/FuNz-0/One-for-More {TEMP_DIR}')
if ret != 0:
    print('Clone gagal!')
else:
    print('Clone berhasil.')
    print('\nIsi folder cdm dari repo baru:')
    for f in sorted(os.listdir(os.path.join(TEMP_DIR, 'cdm'))):
        print(f'  {f}')

Cloning repo CDAD...
Clone berhasil.

Isi folder cdm dari repo baru:
  .ipynb_checkpoints
  __pycache__
  amn.py
  ddim_hacked.py
  func.py
  gpm.py
  hack.py
  logger.py
  mha.py
  model.py
  param.py
  safe_open.py
  sd_amn.py
  vit.py


In [ ]:
import shutil
import os

TEMP_DIR = '/content/cdad_temp'
REF_DIR  = '/content/drive/MyDrive/deep_learning/tubes_DL/referensi'

# Copy semua file cdm yang belum ada di REF_DIR
cdm_src = os.path.join(TEMP_DIR, 'cdm')
cdm_dst = os.path.join(REF_DIR, 'cdm')

print('Menyalin file cdm yang hilang...')
for f in os.listdir(cdm_src):
    src = os.path.join(cdm_src, f)
    dst = os.path.join(cdm_dst, f)
    if not os.path.exists(dst) and os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f'  Disalin: {f}')
    else:
        print(f'  Skip (sudah ada): {f}')

# Copy juga cdad_mvtec.yaml kalau belum ada
models_src = os.path.join(TEMP_DIR, 'models')
models_dst = os.path.join(REF_DIR, 'models')
for f in os.listdir(models_src):
    src = os.path.join(models_src, f)
    dst = os.path.join(models_dst, f)
    if not os.path.exists(dst) and os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f'  Disalin models/{f}')
    else:
        print(f'  Skip models/{f}')

# Verifikasi
print('\nIsi folder cdm sekarang:')
for f in sorted(os.listdir(cdm_dst)):
    print(f'  {f}')

print('\nIsi folder models sekarang:')
for f in sorted(os.listdir(models_dst)):
    print(f'  {f}')

Menyalin file cdm yang hilang...
  Skip (sudah ada): mha.py
  Skip (sudah ada): safe_open.py
  Skip (sudah ada): amn.py
  Skip (sudah ada): func.py
  Skip (sudah ada): sd_amn.py
  Skip (sudah ada): param.py
  Skip (sudah ada): ddim_hacked.py
  Skip (sudah ada): hack.py
  Skip (sudah ada): __pycache__
  Skip (sudah ada): .ipynb_checkpoints
  Skip (sudah ada): logger.py
  Skip (sudah ada): model.py
  Skip (sudah ada): vit.py
  Skip (sudah ada): gpm.py
  Skip models/cdad_visa.yaml
  Skip models/initializer.py
  Skip models/model_helper.py
  Skip models/__init__.py
  Skip models/cdad_mvtec.yaml

Isi folder cdm sekarang:
  .ipynb_checkpoints
  __pycache__
  amn (1).py
  amn (2).py
  amn (3).py
  amn (4).py
  amn.py
  ddim_hacked.py
  func.py
  gpm.py
  hack.py
  logger.py
  mha.py
  model.py
  param.py
  safe_open.py
  sd_amn (1).py
  sd_amn (2).py
  sd_amn (3).py
  sd_amn (4).py
  sd_amn.py
  vit.py

Isi folder models sekarang:
  __init__.py
  __pycache__
  base.ckpt
  cdad_mvtec (1).yaml


In [ ]:
import torch
import sys
import os
import shutil

BASE_PATH = '/content/drive/MyDrive/deep_learning/tubes_DL'
REF_DIR   = os.path.join(BASE_PATH, 'referensi')
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.chdir(REF_DIR)
if REF_DIR not in sys.path:
    sys.path.insert(0, REF_DIR)

cdm_path    = os.path.join(REF_DIR, 'cdm')
models_path = os.path.join(REF_DIR, 'models')
AMN_FILES   = ['amn.py', 'sd_amn.py']
TEMP_DIR    = '/content/cdad_temp'

# Auto-clone jika AMN tidak ada
amn_exists = any(os.path.exists(os.path.join(cdm_path, f)) for f in AMN_FILES)

if not amn_exists:
    print('AMN tidak ditemukan. Auto-clone repo...')
    if os.path.exists(TEMP_DIR):
        os.system(f'rm -rf {TEMP_DIR}')
    ret = os.system(f'git clone https://github.com/FuNz-0/One-for-More {TEMP_DIR}')
    if ret != 0:
        raise RuntimeError('Clone gagal! Cek koneksi internet.')

    # Salin AMN + yaml (selalu overwrite)
    for f in os.listdir(os.path.join(TEMP_DIR, 'cdm')):
        src = os.path.join(TEMP_DIR, 'cdm', f)
        dst = os.path.join(cdm_path, f)
        if os.path.isfile(src) and (f in AMN_FILES or not os.path.exists(dst)):
            shutil.copy2(src, dst)
            print(f'  Disalin: {f}')

    for yml in ['cdad_mvtec.yaml', 'cdad_visa.yaml']:
        src = os.path.join(TEMP_DIR, 'models', yml)
        dst = os.path.join(models_path, yml)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f'  Disalin: {yml}')
    print('Auto-clone selesai.\n')

print('Isi cdm/:', sorted(os.listdir(cdm_path)))
print('Isi models/:', sorted(os.listdir(models_path)))

# Temukan file AMN
amn_path = None
for candidate in AMN_FILES:
    p = os.path.join(cdm_path, candidate)
    if os.path.exists(p):
        amn_path = p
        print(f'\nAMN ditemukan: {candidate}')
        break

if amn_path is None:
    raise FileNotFoundError('AMN masih tidak ada setelah clone. Cek repo GitHub.')

# Patch .cuda() → .to(DEVICE)
with open(amn_path, 'r') as f:
    code = f.read()
if '.cuda()' in code:
    code = code.replace('.cuda()',
        '.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))')
    with open(amn_path, 'w') as f:
        f.write(code)
    print(f'{os.path.basename(amn_path)} di-patch: .cuda() → .to(DEVICE)')
else:
    print(f'{os.path.basename(amn_path)} sudah bersih.')

# Patch yaml
config_path = None
for yml in ['cdad_mvtec.yaml', 'cdad_visa.yaml']:
    p = os.path.join(models_path, yml)
    if os.path.exists(p):
        config_path = p
        print(f'Config: {yml}')
        break

if config_path is None:
    raise FileNotFoundError('Config yaml tidak ditemukan!')

amn_module = os.path.basename(amn_path).replace('.py', '')
with open(config_path, 'r') as f:
    yaml_code = f.read()
yaml_code = yaml_code.replace('cdm.sd_amn', 'cdm.amn')
if amn_module != 'amn':
    yaml_code = yaml_code.replace('cdm.amn', f'cdm.{amn_module}')
with open(config_path, 'w') as f:
    f.write(yaml_code)
print(f'yaml → target AMN: cdm.{amn_module}')

# Bersihkan module cache
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ('cdm', 'ldm', 'taming')):
        del sys.modules[mod]

# Load model CDAD
# Paper Section 4: pre-trained Stable Diffusion v1.5 + VAE (VAE tidak di-finetune)
from cdm.model import create_model

base_ckpt = os.path.join(models_path, 'base.ckpt')
if not os.path.exists(base_ckpt):
    raise FileNotFoundError(f'base.ckpt tidak ada di {base_ckpt}')

print('\nMemuat model CDAD...')
model_cdad = create_model(config_path).to(DEVICE)

weights = torch.load(base_ckpt, map_location='cpu', weights_only=False)
if 'state_dict' in weights:
    weights = weights['state_dict']

sel = {k: v for k, v in weights.items() if 'control_model' not in k}
missing, unexpected = model_cdad.load_state_dict(sel, strict=False)

total     = sum(p.numel() for p in model_cdad.parameters())
trainable = sum(p.numel() for p in model_cdad.parameters() if p.requires_grad)
print(f'  Missing: {len(missing)} | Unexpected: {len(unexpected)}')
print(f'  Total: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M')
print('\nModel CDAD siap!')

Isi cdm/: ['.ipynb_checkpoints', '__pycache__', 'amn (1).py', 'amn (2).py', 'amn (3).py', 'amn (4).py', 'amn.py', 'ddim_hacked.py', 'func.py', 'gpm.py', 'hack.py', 'logger.py', 'mha.py', 'model.py', 'param.py', 'safe_open.py', 'sd_amn (1).py', 'sd_amn (2).py', 'sd_amn (3).py', 'sd_amn (4).py', 'sd_amn.py', 'vit.py']
Isi models/: ['__init__.py', '__pycache__', 'base.ckpt', 'cdad_mvtec (1).yaml', 'cdad_mvtec (2).yaml', 'cdad_mvtec (3).yaml', 'cdad_mvtec (4).yaml', 'cdad_mvtec.yaml', 'cdad_visa.yaml', 'initializer.py', 'model_helper.py']

AMN ditemukan: amn.py
amn.py sudah bersih.
Config: cdad_mvtec.yaml
yaml → target AMN: cdm.amn

Memuat model CDAD...
No module 'xformers'. Proceeding without it.


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/distributed.py:258: LightningDeprecationWarning: `pytorch_lightning.utilities.distributed.rank_zero_only` has been deprecated in v1.8.1 and will be removed in v2.0.0. You can import it from `pytorch_lightning.utilities` instead.
  rank_zero_deprecation(


CDAD: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Loaded model config from [/content/drive/MyDrive/deep_learning/tubes_DL/referensi/models/cdad_mvtec.yaml]
  Missing: 370 | Unexpected: 3
  Total: 1099.4M | Trainable: 892.7M

Model CDAD siap!


In [ ]:
#Definisi iSVD dan Gradient Projection
import torch

def S_and_K(M, threshold=0.999):
    """SVD + k-rank approximation (Algorithm 1 paper)."""
    try:
        U, S, Vh = torch.linalg.svd(M, full_matrices=False)
    except Exception:
        U, S, Vh = torch.svd(M)
    total = (S ** 2).sum()
    if total == 0:
        return U[:, :1]
    ratio      = (S ** 2) / total
    cumulative = torch.cumsum(ratio, dim=0)
    k          = int((cumulative < threshold).sum().item()) + 1
    k          = max(1, min(k, U.shape[1]))
    return U[:, :k]

def iSVD(feature_list, threshold=0.999):
    U_hat = None

    for feat in feature_list:
        # Flatten jika perlu
        if feat.dim() > 2:
            feat = feat.view(feat.size(0), -1)

        M_i = feat.T.cpu().float()  # ambil satu blok saja

        if U_hat is None:
            # Iterasi pertama: hitung langsung dari M_1
            U_hat = S_and_K(M_i, threshold)
        else:
            # Iterasi berikutnya: gabung U_hat (kecil) + M_i (satu blok)
            # U_hat ukurannya d×k (kecil), bukan akumulasi semua M
            combined = torch.cat([U_hat, M_i], dim=1)
            U_hat    = S_and_K(combined, threshold)

        # M_i langsung dibuang setelah dipakai
        # Yang tersimpan hanya U_hat (d×k)
        del M_i

    return U_hat

def project_gradient(grad, U_hat):
    """Gradient Projection (Eq. 7 paper).
    nabla_W_orth = nabla_W - U_hat @ U_hat.T @ nabla_W"""
    if grad is None or U_hat is None:
        return grad
    original_shape = grad.shape
    grad_2d        = grad.view(grad.shape[0], -1).float()
    U              = U_hat.to(grad_2d.device)
    proj           = U @ (U.T @ grad_2d)
    grad_orth      = grad_2d - proj
    return grad_orth.view(original_shape).to(grad.dtype)

print('iSVD dan gradient projection siap.')


iSVD dan gradient projection siap.


In [ ]:
import torch, gc

# Bebaskan semua VRAM yang tidak terpakai
gc.collect()
torch.cuda.empty_cache()

# Aktifkan gradient checkpointing untuk hemat VRAM
# Trade-off: training ~20% lebih lambat tapi memory jauh lebih efisien
if hasattr(model_cdad, 'model') and hasattr(model_cdad.model, 'enable_gradient_checkpointing'):
    model_cdad.model.enable_gradient_checkpointing()

# Freeze VAE — paper Section 4: "VAE does not need further fine-tuning"
# Ini hemat VRAM signifikan karena VAE tidak perlu simpan gradient
for name, param in model_cdad.named_parameters():
    if 'first_stage_model' in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model_cdad.parameters() if p.requires_grad)
print(f'Trainable params setelah freeze VAE: {trainable/1e6:.1f}M')

free, total = torch.cuda.mem_get_info()
print(f'VRAM bebas: {free/1e9:.2f} GB / {total/1e9:.2f} GB')

Trainable params setelah freeze VAE: 892.7M
VRAM bebas: 11.02 GB / 15.64 GB


In [ ]:
#BASE TASK: Training otak
import torch
import torch.nn.functional as F
import os

BASE_PATH  = '/content/drive/MyDrive/deep_learning/tubes_DL'
REF_DIR    = os.path.join(BASE_PATH, 'referensi')
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCH_BASE = 100
LR_BASE    = 1e-5
BATCH_SIZE = 1

os.makedirs('./incre_val/tubes_otak', exist_ok=True)

print('========== BASE TASK: OTAK ==========')
print(f'Epoch      : {EPOCH_BASE}')
print(f'LR         : {LR_BASE}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Data train : {len(train_otak.dataset)} gambar')
print()

optimizer_otak = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_cdad.parameters()),
    lr=LR_BASE
)

# Ambil encoder lokal (Elo) dari AMN untuk hitung L_amn
def get_amn_encoders(model):
    """Kembalikan (local_encoder, nonlocal_encoder) dari AMN jika tersedia."""
    ctrl = getattr(model, 'control_model', None)
    if ctrl is None:
        return None, None
    e_lo  = getattr(ctrl, 'local_encoder',    None)
    e_no  = getattr(ctrl, 'nonlocal_encoder', None)
    if e_lo  is None: e_lo  = getattr(ctrl, 'input_hint_block', None)
    if e_no  is None: e_no  = getattr(ctrl, 'zero_convs',       None)
    return e_lo, e_no

e_lo, e_no = get_amn_encoders(model_cdad)
USE_AMN_LOSS = (e_lo is not None and e_no is not None)
AMN_LAMBDA   = 0.1   # bobot L_amn relatif terhadap L_CDM
print(f'AMN Loss aktif: {USE_AMN_LOSS}')

best_loss_otak = float('inf')
model_cdad.train()

for epoch in range(EPOCH_BASE):
    epoch_loss     = 0.0
    epoch_loss_cdm = 0.0
    epoch_loss_amn = 0.0
    n_batch        = 0

    for batch in train_otak:
        img       = batch['jpg'].to(DEVICE)
        perturbed = batch['hint'].to(DEVICE)
        txt = [' '] * img.shape[0]

        optimizer_otak.zero_grad()

        #  L_CDM: diffusion loss utama
        try:
            loss_cdm, _ = model_cdad.shared_step(
                dict(jpg=img, txt=txt, hint=perturbed)
            )
        except Exception:
            z     = model_cdad.encode_first_stage(img)
            z     = model_cdad.get_first_stage_encoding(z)
            t     = torch.randint(0, model_cdad.num_timesteps,
                                  (z.shape[0],), device=DEVICE)
            noise = torch.randn_like(z)
            z_t   = model_cdad.q_sample(z, t, noise=noise)
            cond  = model_cdad.get_learned_conditioning(
                dict(jpg=img, txt=txt, hint=perturbed)
            )
            noise_pred = model_cdad.apply_model(z_t, t, cond)
            loss_cdm   = F.mse_loss(noise_pred, noise)

        #  L_amn: AMN loss
        # L_amn = || E_no(E_lo(x_tilde) + e_p) - E_lo(x) ||^2
        loss_amn = torch.tensor(0.0, device=DEVICE)
        if USE_AMN_LOSS:
            try:
                with torch.set_grad_enabled(True):
                    c_lo_x      = e_lo(img)
                    c_lo_xtilde = e_lo(perturbed)
                    c_no        = e_no(c_lo_xtilde) if callable(e_no) else c_lo_xtilde
                    if c_no.shape != c_lo_x.shape:
                        c_no = F.interpolate(c_no, size=c_lo_x.shape[-2:],
                                             mode='bilinear', align_corners=False)
                    loss_amn = F.mse_loss(c_no, c_lo_x.detach())
            except Exception:
                loss_amn = torch.tensor(0.0, device=DEVICE)

        loss = loss_cdm + AMN_LAMBDA * loss_amn

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_cdad.parameters(), max_norm=1.0)
        optimizer_otak.step()

        epoch_loss     += loss.item()
        epoch_loss_cdm += loss_cdm.item()
        epoch_loss_amn += loss_amn.item()
        n_batch        += 1

    avg_loss = epoch_loss / max(n_batch, 1)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'  Epoch {epoch+1:3d}/{EPOCH_BASE} — '
              f'Loss: {avg_loss:.4f} '
              f'(CDM: {epoch_loss_cdm/max(n_batch,1):.4f} | '
              f'AMN: {epoch_loss_amn/max(n_batch,1):.4f})')

    if avg_loss < best_loss_otak:
        best_loss_otak = avg_loss
        torch.save(model_cdad.state_dict(), './incre_val/tubes_otak/best.pt')


torch.save(model_cdad.state_dict(), './incre_val/tubes_otak/last.pt')
print(f'\nBase task OTAK selesai! Best loss: {best_loss_otak:.4f}')
print('Lanjut ke Cell 10 untuk hitung iSVD.')


========== BASE TASK: OTAK ==========
Epoch      : 100
LR         : 1e-05
Batch size : 1
Data train : 120 gambar



/content/drive/MyDrive/deep_learning/tubes_DL/referensi/ldm/modules/diffusionmodules/util.py:136: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  torch.cuda.amp.autocast(**ctx.gpu_autocast_kwargs):


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 7.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.32 GiB is allocated by PyTorch, and 93.42 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
#Hitung iSVD dari data otak
import torch
import os

BASE_PATH       = '/content/drive/MyDrive/deep_learning/tubes_DL'
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
THRESHOLD       = 0.999  # paper Section 4.6: γ=0.999
MAX_BATCHES_SVD = 15

print('========== HITUNG iSVD DARI DATA OTAK ==========')
print(f'Threshold iSVD: {THRESHOLD} ')
print()

# Tipe layer yang relevan di U-Net CDAD (ResnetBlock & Transformer)
# significant representation dihitung dari
# layer CNN dan MLP di dalam ResnetBlock dan Transformer U-Net
UNET_TARGET_TYPES = (
    torch.nn.Linear,      # MLP di dalam Transformer block
    torch.nn.Conv2d,      # CNN di dalam ResnetBlock
)

# Keyword nama modul yang menandakan berada di dalam U-Net diffusion model
# Bukan dari VAE (first_stage_model) atau text encoder (cond_stage_model)
UNET_MODULE_KEYWORDS = ('diffusion_model', 'model.diffusion_model')

def is_unet_layer(full_name):
    """Cek apakah layer berasal dari U-Net (bukan VAE / text encoder)."""
    return ('first_stage_model' not in full_name and
            'cond_stage_model'  not in full_name and
            'control_model'     not in full_name)

def hitung_significant_representation(model, train_loader,
                                       threshold=0.999, max_batches=20):
    """Hitung significant representation dari feature U-Net menggunakan iSVD.
    Hook dipasang HANYA pada layer Linear & Conv2d milik U-Net diffusion model,(bukan semua layer model).
    Dipakai untuk gradient projection saat training rahim."""
    collected = {}
    hooks     = []

    def make_hook(name):
        def hook(module, inp, out):
            if name not in collected:
                collected[name] = []
            # Simpan hanya output, langsung ke CPU untuk hemat VRAM
            collected[name].append(out.detach().cpu())
        return hook

    # Pasang hook hanya pada layer U-Net yang relevan
    n_hooked = 0
    for name, module in model.named_modules():
        if isinstance(module, UNET_TARGET_TYPES) and is_unet_layer(name):
            hooks.append(module.register_forward_hook(make_hook(name)))
            n_hooked += 1
    print(f'  Layer di-hook: {n_hooked} (Linear + Conv2d dari U-Net)')

    model.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(train_loader):
            if batch_idx >= max_batches:
                break
            img       = batch['jpg'].to(DEVICE)
            perturbed = batch['hint'].to(DEVICE)
            txt       = [' '] * img.shape[0]
            try:
                z     = model.encode_first_stage(img)
                z     = model.get_first_stage_encoding(z)
                t     = torch.randint(0, model.num_timesteps,
                                      (z.shape[0],), device=DEVICE)
                noise = torch.randn_like(z)
                z_t   = model.q_sample(z, t, noise=noise)
                cond  = model.get_learned_conditioning(
                    dict(jpg=img, txt=txt, hint=perturbed)
                )
                model.apply_model(z_t, t, cond)
            except Exception as e:
                print(f'  Batch {batch_idx} skip: {e}')
            if (batch_idx + 1) % 5 == 0:
                print(f'  Progress: {batch_idx+1}/{max_batches} batch')

    for h in hooks:
        h.remove()

    significant_rep = {}
    for layer_name, feat_list in collected.items():
        if len(feat_list) == 0:
            continue
        try:
            U_hat = iSVD(feat_list, threshold=threshold)
            if U_hat is not None:
                significant_rep[layer_name] = U_hat
        except Exception:
            pass
        # Bebaskan memori setelah diproses
        del feat_list
        collected[layer_name] = []

    print(f'\niSVD selesai. Layer tersimpan: {len(significant_rep)}')
    return significant_rep

sig_rep_otak = hitung_significant_representation(
    model        = model_cdad,
    train_loader = train_otak,
    threshold    = THRESHOLD,
    max_batches  = MAX_BATCHES_SVD
)
print('Lanjutuntuk training incremental rahim.')


In [ ]:
#INCREMENTAL TASK: Training rahim dengan gradient projection
import torch
import torch.nn.functional as F
import os

BASE_PATH         = '/content/drive/MyDrive/deep_learning/tubes_DL'
DEVICE            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCH_INCREMENTAL = 30
LR_INCREMENTAL    = 5e-6
BATCH_SIZE        = 1
AMN_LAMBDA        = 0.1   # bobot L_amn (sama dengan base task)

os.makedirs('./incre_val/tubes_rahim', exist_ok=True)

print('========== INCREMENTAL TASK: RAHIM ==========')
print('Pastikan Cell 10 sudah selesai dulu!')
print(f'Epoch      : {EPOCH_INCREMENTAL}')
print(f'LR         : {LR_INCREMENTAL}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Data train : {len(train_rahim.dataset)} gambar')
print(f'Model      : model_cdad yang sama dari base task otak')
print(f'Anti-forgetting: gradient projection AKTIF')
print()

optimizer_rahim = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_cdad.parameters()),
    lr=LR_INCREMENTAL
)

# Re-ambil encoder AMN
e_lo, e_no = get_amn_encoders(model_cdad)
USE_AMN_LOSS = (e_lo is not None and e_no is not None)
print(f'AMN Loss aktif: {USE_AMN_LOSS}')

best_loss_rahim = float('inf')
model_cdad.train()

for epoch in range(EPOCH_INCREMENTAL):
    epoch_loss     = 0.0
    epoch_loss_cdm = 0.0
    epoch_loss_amn = 0.0
    n_batch        = 0

    for batch in train_rahim:
        img       = batch['jpg'].to(DEVICE)
        perturbed = batch['hint'].to(DEVICE)
        txt       = [' '] * img.shape[0]

        optimizer_rahim.zero_grad()

        #  L_CDM
        try:
            loss_cdm, _ = model_cdad.shared_step(
                dict(jpg=img, txt=txt, hint=perturbed)
            )
        except Exception:
            z     = model_cdad.encode_first_stage(img)
            z     = model_cdad.get_first_stage_encoding(z)
            t     = torch.randint(0, model_cdad.num_timesteps,
                                  (z.shape[0],), device=DEVICE)
            noise = torch.randn_like(z)
            z_t   = model_cdad.q_sample(z, t, noise=noise)
            cond  = model_cdad.get_learned_conditioning(
                dict(jpg=img, txt=txt, hint=perturbed)
            )
            noise_pred = model_cdad.apply_model(z_t, t, cond)
            loss_cdm   = F.mse_loss(noise_pred, noise)

        # L_amn
        loss_amn = torch.tensor(0.0, device=DEVICE)
        if USE_AMN_LOSS:
            try:
                with torch.set_grad_enabled(True):
                    c_lo_x      = e_lo(img)
                    c_lo_xtilde = e_lo(perturbed)
                    c_no        = e_no(c_lo_xtilde) if callable(e_no) else c_lo_xtilde
                    if c_no.shape != c_lo_x.shape:
                        c_no = F.interpolate(c_no, size=c_lo_x.shape[-2:],
                                             mode='bilinear', align_corners=False)
                    loss_amn = F.mse_loss(c_no, c_lo_x.detach())
            except Exception:
                loss_amn = torch.tensor(0.0, device=DEVICE)

        loss = loss_cdm + AMN_LAMBDA * loss_amn

        loss.backward()

        # GRADIENT PROJECTION
        # Proyeksikan gradient rahim ke orthogonal space representasi otak
        # sehingga pengetahuan otak tidak terganggu
        with torch.no_grad():
            for name, param in model_cdad.model.named_parameters():
                if param.grad is not None and name in sig_rep_otak:
                    U_hat = sig_rep_otak[name].to(DEVICE)
                    param.grad.data = project_gradient(param.grad.data, U_hat)

        torch.nn.utils.clip_grad_norm_(model_cdad.parameters(), max_norm=1.0)
        optimizer_rahim.step()

        epoch_loss     += loss.item()
        epoch_loss_cdm += loss_cdm.item()
        epoch_loss_amn += loss_amn.item()
        n_batch        += 1

    avg_loss = epoch_loss / max(n_batch, 1)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'  Epoch {epoch+1:3d}/{EPOCH_INCREMENTAL} — '
              f'Loss: {avg_loss:.4f} '
              f'(CDM: {epoch_loss_cdm/max(n_batch,1):.4f} | '
              f'AMN: {epoch_loss_amn/max(n_batch,1):.4f})')

    if avg_loss < best_loss_rahim:
        best_loss_rahim = avg_loss
        torch.save(model_cdad.state_dict(), './incre_val/tubes_rahim/best.pt')

torch.save(model_cdad.state_dict(), './incre_val/tubes_rahim/last.pt')
print(f'\nIncremental task RAHIM selesai! Best loss: {best_loss_rahim:.4f}')
print('Lanjut ke Cell 12 untuk inference.')


In [ ]:
#Definisi fungsi inference
import torch
import torch.nn.functional as F
import numpy as np
from torchvision import models, transforms

BASE_PATH  = '/content/drive/MyDrive/deep_learning/tubes_DL'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DDIM_STEPS = 10

resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
resnet = resnet.to(DEVICE)
resnet.eval()

feat_cache = {}
def make_feat_hook(name):
    def hook(module, inp, out):
        feat_cache[name] = out
    return hook

resnet.layer2.register_forward_hook(make_feat_hook('layer2'))
resnet.layer3.register_forward_hook(make_feat_hook('layer3'))

imagenet_norm = transforms.Normalize(
    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
)

def hitung_anomaly_score(x_hat, x_tilde, target_size=256):
    """
    M^l = || phi^l(x_hat) - phi^l(x_tilde) ||^2
    S   = sum_l Upsample(M^l), score = max(S)"""
    def to_imagenet(x):
        return imagenet_norm((x.clamp(-1, 1) + 1) / 2)
    with torch.no_grad():
        resnet(to_imagenet(x_hat))
        feat_hat = {k: v.clone() for k, v in feat_cache.items()}
        resnet(to_imagenet(x_tilde))
        feat_tilde = {k: v.clone() for k, v in feat_cache.items()}
    score_map = None
    for layer in ['layer2', 'layer3']:
        diff     = (feat_hat[layer] - feat_tilde[layer]) ** 2
        dist_map = diff.sum(dim=1, keepdim=True)
        dist_up  = F.interpolate(dist_map, size=(target_size, target_size),
                                  mode='bilinear', align_corners=False)
        score_map = dist_up if score_map is None else score_map + dist_up
    score_map_np  = score_map.squeeze().cpu().numpy()
    return float(score_map_np.max()), score_map_np

@torch.no_grad()
def inference_satu_gambar(model, img_tensor, ddim_steps=10):
    """Inference """
    model.eval()
    img   = img_tensor.to(DEVICE)
    txt   = [' '] * img.shape[0]
    z0    = model.encode_first_stage(img)
    z0    = model.get_first_stage_encoding(z0)
    x_hat = model.decode_first_stage(z0)
    t     = torch.tensor([model.num_timesteps - 1], device=DEVICE)
    zT    = model.q_sample(z0, t, noise=torch.randn_like(z0))
    cond  = model.get_learned_conditioning(dict(jpg=img, txt=txt, hint=img))
    try:
        from ldm.models.diffusion.ddim import DDIMSampler
        sampler  = DDIMSampler(model)
        z_hat, _ = sampler.sample(
            S=ddim_steps, conditioning=cond,
            batch_size=img.shape[0], shape=z0.shape[1:],
            verbose=False, x_T=zT
        )
    except Exception as e:
        print(f'  DDIM fallback: {e}')
        z_hat = model.p_sample_loop(
            cond=cond, shape=zT.shape,
            return_intermediates=False, x_T=zT
        )
    x_tilde          = model.decode_first_stage(z_hat)
    score, score_map = hitung_anomaly_score(x_hat, x_tilde)
    return x_hat, x_tilde, score, score_map

print('Fungsi inference siap.')


In [ ]:
#Evaluasi AUROC
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import torch

BASE_PATH  = '/content/drive/MyDrive/deep_learning/tubes_DL'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DDIM_STEPS = 10

def evaluasi(model, test_loader, nama_task, ddim_steps=10):
    print(f'\n{"="*50}')
    print(f'EVALUASI: {nama_task}')
    print(f'{"="*50}')
    print(f'Kelas: {test_loader.dataset.classes}')

    all_scores, all_labels = [], []
    model.eval()

    for i, (img, label) in enumerate(test_loader):
        try:
            _, _, score, _ = inference_satu_gambar(model, img.to(DEVICE), ddim_steps)
            all_scores.append(score)
            all_labels.append(int(label.item()))
        except Exception as e:
            print(f'  Skip gambar {i}: {e}')
        if (i + 1) % 50 == 0:
            print(f'  Progress: {i+1}/{len(test_loader)}')

    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)
    if all_scores.max() > all_scores.min():
        all_scores = (all_scores - all_scores.min()) / (all_scores.max() - all_scores.min())

    if len(set(all_labels)) < 2:
        print('Peringatan: hanya satu kelas, AUROC tidak valid.')
        return None

    auroc = roc_auc_score(all_labels, all_scores)
    print(f'\nA-AUROC {nama_task}: {auroc*100:.2f}%')

    fpr, tpr, _ = roc_curve(all_labels, all_scores)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'AUROC = {auroc*100:.2f}%')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve — {nama_task}')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'./incre_val/tubes_{nama_task.lower()}/roc_curve.png', dpi=150)
    plt.show()
    return auroc, all_scores, all_labels

result_otak  = evaluasi(model_cdad, test_otak,  'OTAK',  ddim_steps=DDIM_STEPS)
result_rahim = evaluasi(model_cdad, test_rahim, 'RAHIM', ddim_steps=DDIM_STEPS)

if result_otak and result_rahim:
    print(f'\n{"="*50}')
    print('RINGKASAN HASIL AKHIR')
    print(f'{"="*50}')
    print(f'  A-AUROC OTAK  : {result_otak[0]*100:.2f}%')
    print(f'  A-AUROC RAHIM : {result_rahim[0]*100:.2f}%')
    print(f'  Rata-rata     : {(result_otak[0]+result_rahim[0])/2*100:.2f}%')


In [ ]:
#Visualisasi heatmap anomali
import matplotlib.pyplot as plt
import numpy as np
import torch

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DDIM_STEPS = 10

def visualisasi_prediksi(model, test_loader, nama_task, n_contoh=5, ddim_steps=10):
    print(f'Visualisasi — {nama_task}')
    model.eval()
    fig, axes = plt.subplots(n_contoh, 4, figsize=(14, 3 * n_contoh))
    if n_contoh == 1:
        axes = axes[np.newaxis, :]

    count = 0
    for img, label in test_loader:
        if count >= n_contoh:
            break
        try:
            x_hat, x_tilde, score, score_map = inference_satu_gambar(
                model, img.to(DEVICE), ddim_steps=ddim_steps)
        except Exception as e:
            print(f'  Skip: {e}')
            continue

        def to_np(t):
            arr = t.squeeze().cpu().float()
            arr = (arr.clamp(-1, 1) + 1) / 2
            return arr.permute(1, 2, 0).numpy()

        hmap = score_map.copy()
        if hmap.max() > hmap.min():
            hmap = (hmap - hmap.min()) / (hmap.max() - hmap.min())

        label_str = test_loader.dataset.classes[int(label.item())]
        axes[count, 0].imshow(to_np(img));      axes[count, 0].set_title(f'Input\n({label_str})', fontsize=9);        axes[count, 0].axis('off')
        axes[count, 1].imshow(to_np(x_hat));    axes[count, 1].set_title('Rekonstruksi VAE\n(x_hat)', fontsize=9);    axes[count, 1].axis('off')
        axes[count, 2].imshow(to_np(x_tilde));  axes[count, 2].set_title('Rekonstruksi Diffusion\n(x_tilde)', fontsize=9); axes[count, 2].axis('off')
        axes[count, 3].imshow(hmap, cmap='jet'); axes[count, 3].set_title(f'Heatmap Anomali\nScore:{score:.3f}', fontsize=9); axes[count, 3].axis('off')
        count += 1

    plt.suptitle(f'Prediksi CDAD — {nama_task}', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'./incre_val/tubes_{nama_task.lower()}/visualisasi.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Disimpan di ./incre_val/tubes_{nama_task.lower()}/visualisasi.png')

visualisasi_prediksi(model_cdad, test_otak,  'OTAK',  n_contoh=5, ddim_steps=DDIM_STEPS)
visualisasi_prediksi(model_cdad, test_rahim, 'RAHIM', n_contoh=5, ddim_steps=DDIM_STEPS)
